In [1]:
import os
import requests
from pyvis.network import Network

In [ ]:
DATASET_ID = "6dvr-xwnh"
BASE_URL = f"https://data.cityofchicago.org/api/v3/views/{DATASET_ID}/query.json"
APP_TOKEN = os.environ.get("SOCRATA_APP_TOKEN")

if not APP_TOKEN:
    raise RuntimeError("SOCRATA_APP_TOKEN is not set. SODA3 requests require identification (app token or auth).")

headers = {
    "X-App-Token": APP_TOKEN,
    "Accept": "application/json",
}

# SoQL query (same idea as your SODA2 $select/$group, but expressed as one query string)
soql = """
SELECT
  pickup_community_area,
  dropoff_community_area,
  count(*) AS trips
WHERE
  pickup_community_area IS NOT NULL
  AND dropoff_community_area IS NOT NULL
  AND trip_start_timestamp >= '2025-01-01T00:00:00.000'
  AND trip_start_timestamp <  '2025-01-02T00:00:00.000'
GROUP BY
  pickup_community_area,
  dropoff_community_area
"""

payload = {
    "query": soql,
    "page": {"pageNumber": 1, "pageSize": 10000},   # plenty; max is 5929 pairs
    "includeSynthetic": False,
}

resp = requests.post(BASE_URL, headers=headers, json=payload, timeout=(10, 300))
if resp.status_code != 200:
    print("Status:", resp.status_code)
    print("Response text:", resp.text)   # <-- this will usually explain the 400
    resp.raise_for_status()
out = resp.json()

# Different portals/versions may wrap results; handle the common cases defensively
rows = out.get("results") or out.get("data") or out

ride_net = Network(notebook=True, cdn_resources="in_line", directed=True)

for row in rows:
    # v3 often returns typed values, but be defensive (strings can still appear)
    pickup = int(float(row["pickup_community_area"]))
    dropoff = int(float(row["dropoff_community_area"]))
    trips = int(row["trips"])

    ride_net.add_node(pickup, label=str(pickup), title=f"Chicago Community Area {pickup}")
    ride_net.add_node(dropoff, label=str(dropoff), title=f"Chicago Community Area {dropoff}")
    ride_net.add_edge(pickup, dropoff, arrows="to", value=trips, title=f"Trips: {trips:,}")

file_path = os.path.abspath("rideshare_edges_aggregated.html")
ride_net.write_html(file_path)
print(file_path)


Status: 200
Response text (first 500 chars): [
{"trip_id":"00075dcc3155595afaf709e4d2646135f8827cd8","trip_start_timestamp":"2025-12-31T23:45:00.000","trip_end_timestamp":"2026-01-01T00:00:00.000","trip_seconds":"511","trip_miles":"0.3147","percent_time_chicago":"0.998","percent_distance_chicago":"0.9965","pickup_census_tract":"17031280100","dropoff_census_tract":"17031081800","pickup_community_area":"28","dropoff_community_area":"8","fare":"10","tip":"0","additional_charges":"5.41","trip_total":"15.41","shared_trip_authorized":false,"shar
Parsed JSON keys: <class 'list'>
